In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [4]:
def build_prompt(word):
    return f"""
You are a Korean language tutor helping intermediate learners (TOPIK Level 3).

A student provides a SINGLE Korean word.

Your job is to generate:

1. Exactly 3 example sentences using the word
2. English translations
3. Grammar breakdown for each sentence
4. Usage notes

Rules:
- Sentences must be natural and useful in daily life
- Use intermediate grammar (TOPIK 3 level)
- Highlight grammar patterns (e.g., -고 싶다, -아/어서)
- Keep explanations concise

Word: {word}

Output format:

1.
Sentence:
Translation:
Grammar:
Explanation:

2.
...

3.
...

Usage Notes:
"""

In [5]:
def generate_response(word):
    prompt = build_prompt(word)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

In [6]:
word = "먹다"
print(generate_response(word))


You are a Korean language tutor helping intermediate learners (TOPIK Level 3).

A student provides a SINGLE Korean word.

Your job is to generate:

1. Exactly 3 example sentences using the word
2. English translations
3. Grammar breakdown for each sentence
4. Usage notes

Rules:
- Sentences must be natural and useful in daily life
- Use intermediate grammar (TOPIK 3 level)
- Highlight grammar patterns (e.g., -고 싶다, -아/어서)
- Keep explanations concise

Word: 먹다

Output format:

1.
Sentence:
Translation:
Grammar:
Explanation:

2.
...

3.
...

Usage Notes:

---

**Word: 먹다 (to eat)**

1.
Sentence: 점심을 먹었어요.
Translation: I ate lunch.
Grammar: 점심을 (object particle) + 먹었어요 (past tense of the verb 먹다).
Explanation: The object particle 을 is attached to 점심 (lunch), and 먹었어요 is the past tense form of 먹다, indicating a completed action.

2.
Sentence: 저는 매일 아침에 빵을 먹고 싶어요.
Translation: I want to eat bread every morning.
Grammar: 저는 (subject marker) + 매일 (every day) + 아침에 (every morning) + 빵을 (object

In [7]:
def clean_output(text, word):
    return text.split(f"Word: {word}")[-1].strip()

In [9]:
while True:
    word = input("Enter a Korean word (or 'quit'): ")

    if word.lower() == "quit":
        break

    result = generate_response(word)
    print("\n" + result + "\n")

Enter a Korean word (or 'quit'): 달리다


You are a Korean language tutor helping intermediate learners (TOPIK Level 3).

A student provides a SINGLE Korean word.

Your job is to generate:

1. Exactly 3 example sentences using the word
2. English translations
3. Grammar breakdown for each sentence
4. Usage notes

Rules:
- Sentences must be natural and useful in daily life
- Use intermediate grammar (TOPIK 3 level)
- Highlight grammar patterns (e.g., -고 싶다, -아/어서)
- Keep explanations concise

Word: 달리다

Output format:

1.
Sentence:
Translation:
Grammar:
Explanation:

2.
...

3.
...

Usage Notes:
- When to use this word
- Common verbs or expressions that go with 달리다
- Tips for pronunciation

Solution 1:

1.
Sentence: 그는 점심을 먹고 달리다.
Translation: He eats lunch and runs.
Grammar: 달리다 (verb stem + 다) + 고 (conjunction) + 점심을 (object)
Explanation: The verb stem 달린다 is used here in its infinitive form with 다 added at the end to create a dictionary form. The conjunction 고 is used to connect two act